In [1]:
#@title <h1 style="font-size:88px;">初始化</h1>

# 安装 Git
!apt-get install -y git

# 克隆 TikTokLive 仓库
!git clone https://github.com/isaackogan/TikTokLive.git

# 切换到 TikTokLive 目录
%cd TikTokLive

# 安装指定版本的 pyee
!pip install pyee==11.1.0

# 安装 TikTokLive
!pip install TikTokLive

!pip install pandas matplotlib

!pip install dash

!pip install plotly pandas

!pip install pyngrok

!pip install --upgrade git+https://github.com/isaackogan/TikTokLive.git@master

# 使用 IPython.display 来清除输出
from IPython.display import clear_output

# 清除输出
clear_output(wait=True)

# 输出初始化成功的消息
print("初始化成功")

初始化成功


In [ ]:
# 1) 安装依赖（如果还没安装的话）
# apt-get update && apt-get install -y git
# pip install pyee==11.1.0 TikTokLive pandas matplotlib dash plotly
# （pyngrok 已无需使用，可以不再安装）

import asyncio
import time
import threading
from TikTokLive import TikTokLiveClient
from TikTokLive.events import ConnectEvent, RoomUserSeqEvent
from TikTokLive.client.logger import LogLevel

import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html
from dash.dependencies import Input, Output

###################################
# 多主播监控逻辑
###################################
user_list = ["@nikonikodiy_shinano", "@fancynana14", "@njkgdugp0bz"]
monitor_data = {uid: [] for uid in user_list}
total_viewers_data = {uid: 0 for uid in user_list}
clients = []

def create_client(user_id: str) -> TikTokLiveClient:
    client = TikTokLiveClient(unique_id=user_id)
    client.logger.setLevel(LogLevel.INFO.value)

    @client.on(ConnectEvent)
    async def on_connect(event: ConnectEvent):
        print(f"[{user_id}] Connected!")

    @client.on(RoomUserSeqEvent)
    async def on_room_user_seq(event: RoomUserSeqEvent):
        data = event.__dict__
        current_viewers = data.get("total")
        total_viewers = data.get("total_user")

        print(f"[{user_id}] RoomUserSeqEvent: current = {current_viewers}, total = {total_viewers}")

        if current_viewers is not None:
            monitor_data[user_id].append((time.time(), current_viewers))

        if total_viewers is not None:
            total_viewers_data[user_id] = total_viewers

    return client

for uid in user_list:
    clients.append(create_client(uid))

async def check_loop(client: TikTokLiveClient):
    while True:
        try:
            if not await client.is_live():
                print(f"[{client.unique_id}] Not live. Checking again in 60s.")
                await asyncio.sleep(60)
            else:
                print(f"[{client.unique_id}] Live detected, connecting...")
                await client.connect()
            await asyncio.sleep(10)
        except Exception as e:
            print(f"[{client.unique_id}] Error: {e}, sleep 60s then retry")
            await asyncio.sleep(60)

async def main_all():
    tasks = [asyncio.create_task(check_loop(c)) for c in clients]
    await asyncio.gather(*tasks)

def run_monitor_loop():
    asyncio.run(main_all())

def start_monitoring():
    t = threading.Thread(target=run_monitor_loop, daemon=True)
    t.start()

###################################
# Dash
###################################
app = Dash(__name__)

app.layout = html.Div([
    html.H1("TikTok Live Viewer Dashboard"),
    html.Div(
        id="graphs-container",
        children=[
            html.Div(
                [
                    html.H2(f"Live Data for {uid}"),
                    dcc.Graph(id=f"graph-{uid}"),
                    html.Div(id=f"total-viewers-{uid}", style={"fontSize": "20px", "marginTop": "10px"})
                ],
                style={"border": "1px solid black", "padding": "10px", "marginBottom": "20px"}
            )
            for uid in user_list
        ]
    ),
    dcc.Interval(
        id='interval-component',
        interval=10 * 1000,  # 每10秒刷新一次
        n_intervals=0
    )
])

@app.callback(
    [Output(f"graph-{uid}", "figure") for uid in user_list] +
    [Output(f"total-viewers-{uid}", "children") for uid in user_list],
    [Input("interval-component", "n_intervals")]
)
def update_graphs_and_totals(n):
    figures = []
    totals = []
    for uid in user_list:
        points = monitor_data[uid]
        cutoff_time = time.time() - 5 * 60
        filtered_points = [(ts, v) for ts, v in points if ts >= cutoff_time]
        monitor_data[uid] = filtered_points

        if not filtered_points:
            figures.append(px.line())
        else:
            df = pd.DataFrame(filtered_points, columns=["time", "viewers"])
            df["time_str"] = df["time"].apply(lambda x: time.strftime("%H:%M:%S", time.localtime(x)))
            fig = px.line(df, x="time_str", y="viewers", title=f"Live Viewer Count for {uid}")
            fig.update_layout(yaxis_title="Current Viewers")
            figures.append(fig)

        total_viewers = total_viewers_data.get(uid, 0)
        totals.append(f"Total Viewers: {total_viewers}")

    return figures + totals

def run_dash():
    # 无需使用 pyngrok，直接在 0.0.0.0:8050 启动
    app.run_server(host="0.0.0.0", port=6006, debug=False, use_reloader=False)

# 启动监控
start_monitoring()

# 在子线程启动 Dash
dash_thread = threading.Thread(target=run_dash, daemon=True)
dash_thread.start()

# 主线程不退出
while True:
    time.sleep(1)


[nikonikodiy_shinano] Error: , sleep 60s then retry
[fancynana14] Error: , sleep 60s then retry
[njkgdugp0bz] Error: , sleep 60s then retry


In [ ]:
#@title <h1 style="font-size:88px;">初始化</h1>

# 安装 Git
!apt-get install -y git

# 克隆 TikTokLive 仓库
!git clone https://github.com/isaackogan/TikTokLive.git

# 切换到 TikTokLive 目录
%cd TikTokLive

# 安装指定版本的 pyee
!pip install pyee==11.1.0

# 安装 TikTokLive
!pip install TikTokLive

!pip install pandas matplotlib

!pip install dash

!pip install plotly pandas

!pip install pyngrok

!pip install --upgrade git+https://github.com/isaackogan/TikTokLive.git@master

# 使用 IPython.display 来清除输出
from IPython.display import clear_output

# 清除输出
clear_output(wait=True)

# 输出初始化成功的消息
print("初始化成功")

初始化成功


In [ ]:
#@title <h1 style="font-size:88px;">初始化</h1>

# 安装 Git
!apt-get install -y git

# 克隆 TikTokLive 仓库
!git clone https://github.com/isaackogan/TikTokLive.git

# 切换到 TikTokLive 目录
%cd TikTokLive

# 安装指定版本的 pyee
!pip install pyee==11.1.0

# 安装 TikTokLive
!pip install TikTokLive

!pip install pandas matplotlib

!pip install dash

!pip install plotly pandas

!pip install pyngrok

!pip install --upgrade git+https://github.com/isaackogan/TikTokLive.git@master

# 使用 IPython.display 来清除输出
from IPython.display import clear_output

# 清除输出
clear_output(wait=True)

# 输出初始化成功的消息
print("初始化成功")

初始化成功


In [ ]:
#@title <h1 style="font-size:88px;">初始化</h1>

# 安装 Git
!apt-get install -y git

# 克隆 TikTokLive 仓库
!git clone https://github.com/isaackogan/TikTokLive.git

# 切换到 TikTokLive 目录
%cd TikTokLive

# 安装指定版本的 pyee
!pip install pyee==11.1.0

# 安装 TikTokLive
!pip install TikTokLive

!pip install pandas matplotlib

!pip install dash

!pip install plotly pandas

!pip install pyngrok

!pip install --upgrade git+https://github.com/isaackogan/TikTokLive.git@master

# 使用 IPython.display 来清除输出
from IPython.display import clear_output

# 清除输出
clear_output(wait=True)

# 输出初始化成功的消息
print("初始化成功")

初始化成功
